haven't tested out this code yet. need to map features to patches. currently have one .h5 per slide containing features for all patches. code belows assumes patches are in order and maps them by index. might need to map by coordinates. check coords column of h5 file and patch name from numpy file name to ensure mapping is correct. 

In [ ]:
import os
import h5py
import pandas as pd

# Set your h5 folder path and desired output CSV path
h5_folder = r'C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_10x\uni\h5_files'
output_csv = r'C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_10x\uni\patch_features_metadata.csv'

data = []
label_map = {'FA': 0, 'PT': 1}

for fname in os.listdir(h5_folder):
    if not fname.endswith('.h5'):
        continue
    slide_id = fname.replace('.h5', '')
    slide_path = os.path.join(h5_folder, fname)
    try:
        with h5py.File(slide_path, 'r') as f:
            features = f['features'][:]
            for i in range(features.shape[0]):
                patch_id = f"{slide_id}_patch{i}"
                label = None
                for prefix, cls in label_map.items():
                    if slide_id.startswith(prefix):
                        label = cls
                        break
                if label is None:
                    print(f"Warning: Could not determine label for {slide_id}")
                    continue
                data.append({'slide_id': slide_id, 'patch_id': patch_id, 'label': label, 'h5_path': slide_path, 'index': i})
    except Exception as e:
        print(f"❌ Error reading {fname}: {e}")

df = pd.DataFrame(data)
df.to_csv(output_csv, index=False)
print(f"✅ Metadata saved to {output_csv}")
